In [2]:
import gymnasium as gym
import numpy as np
import time

In [54]:
env = gym.make('CartPole-v1', render_mode='human')
state= env.reset()
print(state)
env.render()

(array([-0.01640498, -0.02180126,  0.02844888,  0.01868777], dtype=float32), {})


In [60]:
for i in range(40):
    env.step(i%2)
    time.sleep(0.1)

env.close()

/home/aaron-dsouza/programming/miniconda3/envs/mujoco/lib/python3.12/site-packages/gymnasium/envs/classic_control/cartpole.py:213: UserWarning: WARN: You are calling 'step()' even though this environment has already returned terminated = True. You should always call 'reset()' once you receive 'terminated = True' -- any further steps are undefined behavior.
  logger.warn(


In [ ]:
# simulate the environment
episodeNumber = 10
timesteps = 100
for episodeIndex in range(episodeNumber):
    initial_state= env.reset()
    env.render()
    for timeIndex in range(timesteps):
        rand_action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(rand_action)
        time.sleep(0.01)
        if(terminated):
            time.sleep(1) #increase to see the separate episodes
            break

env.close()

<bound method Wrapper.close of <TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>>

In [3]:
class Q_Learning:

    def __init__(self, env, alpha, gamma, epsilon, numberEpisodes, numberOfBins, lowerBounds, upperBounds):

        self.env = env
        self.alpha=alpha
        self.gamma=gamma
        self.epsilon=epsilon
        self.actionNumber = env.action_space.n
        self.numberEpisodes = numberEpisodes
        self.numberOfBins = numberOfBins
        self.lowerBounds = lowerBounds
        self.upperBounds = upperBounds

        self.sumRewardsEpisode=[]
        self.Qmatrix = np.random.uniform(low=0,high=1, 
                                         size = tuple(numberOfBins) + (self.actionNumber,))
        
    def returnIndexState(self,state):
        cartPositionBin = np.linspace(self.lowerBounds[0], self.upperBounds[0], self.numberOfBins[0])
        cartVelocityBin = np.linspace(self.lowerBounds[1], self.upperBounds[1], self.numberOfBins[1])
        poleAngleBin = np.linspace(self.lowerBounds[2], self.upperBounds[2], self.numberOfBins[2])
        poleAngleVelocityBin = np.linspace(self.lowerBounds[3], self.upperBounds[3], self.numberOfBins[3])

        indexPosition = np.maximum(np.digitize(state[0], cartPositionBin)-1, 0)
        indexVelocity=np.maximum(np.digitize(state[1],cartVelocityBin)-1,0)
        indexAngle=np.maximum(np.digitize(state[2],poleAngleBin)-1,0)
        indexAngularVelocity=np.maximum(np.digitize(state[3],poleAngleVelocityBin)-1,0)

        return (indexPosition, indexVelocity, indexAngle, indexAngularVelocity)
    
    def selectAction(self,state,index):
        if index < 500:
            return np.random.choice(self.actionNumber)

        randomNumber = np.random.random()
        if index > 700:
            self.epsilon = 0.999 * self.epsilon

        if randomNumber < self.epsilon:
            return np.random.choice(self.actionNumber)

        else:
            q_values = self.Qmatrix[self.returnIndexState(state)]
            best_actions = np.where(q_values == np.max(q_values))[0]
            return np.random.choice(best_actions)

    def simulateEpisodes(self):
        for indexEpisode in range(self.numberEpisodes):
            rewardsEpisode = []
            (stateS,_) = self.env.reset()
            stateS = list(stateS)
            terminalState = False
            while not terminalState:
                stateSIndex = self.returnIndexState(stateS)
                actionA = self.selectAction(stateS, indexEpisode)
                stateSprime, reward, terminalState, _, _ = self.env.step(actionA)
                rewardsEpisode.append(reward)
                stateSprime = list(stateSprime)
                stateSprimeIndex = self.returnIndexState(stateSprime)
                QmaxPrime = np.max(self.Qmatrix[stateSprimeIndex])
                if not terminalState:
                    target = reward + self.gamma * QmaxPrime
                else:
                    target = reward

                error = target - self.Qmatrix[stateSIndex + (actionA,)]
                self.Qmatrix[stateSIndex + (actionA,)] += self.alpha * error

                stateS = stateSprime
            #print("Sum of rewards {}".format(np.sum(rewardsEpisode)))        
            self.sumRewardsEpisode.append(np.sum(rewardsEpisode))

    def simulateLearnedStrategy(self):
        env1 = gym.make('CartPole-v1', render_mode='human')
        (currentState, _) = env1.reset()
        env1.render()
        timeSteps=1000
        obtainedRewards = []

        for timeIndex in range(timeSteps):
            q_values = self.Qmatrix[self.returnIndexState(currentState)]
            best_actions = np.where(q_values == np.max(q_values))[0]
            actionInStateS= np.random.choice(best_actions)
            currentState, reward, terminated, truncated, info = env1.step(actionInStateS)
            obtainedRewards.append(reward)
            time.sleep(0.05)
            if(terminated):
                time.sleep(1)
                break
        return obtainedRewards, env1
    
    def simulateRandomStrategy(self):
        env2=gym.make('CartPole-v1')
        (currentState,_)=env2.reset()
        env2.render()
        # number of simulation episodes
        episodeNumber=100
        # time steps in every episode
        timeSteps=1000
        # sum of rewards in each episode
        sumRewardsEpisodes=[]
         
         
        for episodeIndex in range(episodeNumber):
            rewardsSingleEpisode=[]
            initial_state=env2.reset()
            print(episodeIndex)
            for timeIndex in range(timeSteps):
                random_action=env2.action_space.sample()
                observation, reward, terminated, truncated, info =env2.step(random_action)
                rewardsSingleEpisode.append(reward)
                if (terminated):
                    break     
            sumRewardsEpisodes.append(np.sum(rewardsSingleEpisode))
        return sumRewardsEpisodes,env2

In [4]:
import matplotlib.pyplot as plt 
 
#env=gym.make('CartPole-v1',render_mode='human')
env=gym.make('CartPole-v1')
(state,_)=env.reset()
#env.render()
#env.close()
 
# here define the parameters for state discretization
upperBounds=env.observation_space.high
lowerBounds=env.observation_space.low
cartVelocityMin=-3
cartVelocityMax=3
poleAngleVelocityMin=-10
poleAngleVelocityMax=10
upperBounds[1]=cartVelocityMax
upperBounds[3]=poleAngleVelocityMax
lowerBounds[1]=cartVelocityMin
lowerBounds[3]=poleAngleVelocityMin
 
numberOfBinsPosition=30
numberOfBinsVelocity=30
numberOfBinsAngle=30
numberOfBinsAngleVelocity=30
numberOfBins=[numberOfBinsPosition,numberOfBinsVelocity,numberOfBinsAngle,numberOfBinsAngleVelocity]
 
# define the parameters
alpha=0.1
gamma=1
epsilon=0.2
numberEpisodes=15000 
 
# create an object
Q1=Q_Learning(env,alpha,gamma,epsilon,numberEpisodes,numberOfBins,lowerBounds,upperBounds)
# run the Q-Learning algorithm
Q1.simulateEpisodes()
# simulate the learned strategy



In [5]:
(obtainedRewardsOptimal,env1)=Q1.simulateLearnedStrategy()
 
# close the environment
env1.close()

In [6]:
for i in range(10):
    (obtainedRewardsOptimal,env1)=Q1.simulateLearnedStrategy()

env1.close()

KeyboardInterrupt: 